<div align="center">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabianalexisbernabecruz-code/portafolio-actuaria-unam/blob/main/06_Neural_Network_Sales_Prediction.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-181717?logo=github)](https://github.com/fabianalexisbernabecruz-code/portafolio-actuaria-unam)

</div>



<font face="Times New Roman">

 <img src="https://upload.wikimedia.org/wikipedia/commons/c/ca/Escudo-UNAM-escalable.svg" width="150" height="170" align="left"/>

  <img src="https://upload.wikimedia.org/wikipedia/commons/5/5b/Escudo-facultad-ciencias-unam-escalable.svg" width="150" height="170" align="right"/>

<h5 align="center"><i> </i></h5>
<h1 align="center"><b>Redes Neuronales para Predicción de Ventas</b></h1>
</font>
<font face="Times New Roman">

---
---

<h2 align="center"><i>Modelo MLP Shallow en PyTorch</i></h2>
<h5 align="center"><i>Diseño, entrenamiento y evaluación de una red neuronal para predecir ventas de limonada.</i></h5>

---
---

<h5 align="center"><i>Universidad Nacional Autónoma de México</i></h5>
<h5 align="center"><i>Facultad de Ciencias — Actuaría</i></h5>
<h5 align="center"><i>Materia: Manejo de Datos</i></h5>

---
---

<font color="Black" face="Times New Roman">
<h5 align="center"><i>Bernabe Cruz Fabian Alexis</i></h5>

</font>
</font>

<font color="Teal" face="Times New Roman">
<h1 align="center" style><b>Proyecto Final. Redes neuronales.</b></h1>
</font>

<font color="Black" face="Times New Roman">
<h5 align="center"><i>
<h5 align="center"><i>
<h5 align="center"><i>Predicción de ventas usando una red neuronal </i></h5>
<h5 align="center"><i>Bernabe Cruz Fabian Alexis (FABC)</i></h5>

<font face="Times New Roman">

---

</font>

<font face="Times New Roman">

# **Descripción del proyecto**

</font>

<font face="Times New Roman">

El proyecto consiste en extraer datos de la base de datos "LIMONADA" que se encuentra en el servidor del curso mismos con los que vamos a entrenar una red neuronal cuyo propósito será el de predecir el comportamiento de las ventas de limonadas, en particular el apartado "vendidos"; dados los demás parámetros de la tabla "VENTAS".

Este proyecto está guiado y tiene como base el tutorial [Predict Sales Using a Neural Network](https://www.leaky.ai/projectlemonade).

</font>

<font face="Times New Roman">

**Introducción**

El problema de conocer las ventas futuras de una empresa es un problema que atañe a estos organismos lucrativos de forma directa pues, como es sabido, el fin de una empresa lucrativa es maximizar sus ganancias y minimizar sus costos, es en este sentido en que una red neuronal es una herramienta de gran utilidad puesto que con ella se es capaz de,  con su predicción luego de haberla entrenado con datos históricos, optimizar los inventarios de la empresa, prever la demanda lo permite a las empresas mantener niveles de inventario adecuados, cambiar sus estrategias de venta de la empresa permitiendo con ello flujos más grandes de capital y con esto cumplir los objetivos de las empresas.

</font>

<font face="Times New Roman">

**¿Cómo se hará?**

Se implementará una pequeña red neuronal que tomará datos etiquetados de una base de datos y con estos se entrenará para que sea capaz de aprender (aprendizaje supervisado) y con ello predecir las ventas de la empresa.

</font>

<font face="Times New Roman">

# **Sección de bibliotecas**

</font>

In [ ]:
# Para evitar ver las advertencias que nos muestra python al instalar
import warnings
# Instalamos el conector
!pip install mysql-connector-python
warnings.filterwarnings('ignore')

In [ ]:
# torch nos ayudará con la parte del deep learning y en particular los tensores y parte
# matricial, vectorial que se requiere para el aprendizaje de la red neuronal
import torch
# Pandas para la parte de análisis de datos
import pandas as pd
# para visualizar los datosen gráficos
import matplotlib.pyplot as plt
# Para evitar ver las advertencias que nos muestra python al instalar
import warnings
# Para conectarnos a la base desde python
import mysql.connector
# Para la creación de un documento en donde registraremos nuestras predicciones arrojadas por la red neuronal
import json

<font face="Times New Roman">

# **Conexión a la base**

</font>

<font face="Times New Roman">

Dada la descripción lo primero que haremos será establecer conexión con la base de datos con la siguiente información:

*   host: hpc-matematicas-z.fciencias.unam.mx
*   usuario: JimmyNeutron
*   pasword: CarlSheen
*   Base de Datos: LIMONADA

Nos conectamos a la base de datos desde Python usando el connector de mysql

</font>

In [ ]:
# Conexión a la base de datos con manejo de excepción para Google Colab / entornos externos
import os, pandas as pd
try:
    print('Intentando conectar a MySQL UNAM (hpc-matematicas-z.fciencias.unam.mx)...')
    con = mysql.connector.connect(
        host='hpc-matematicas-z.fciencias.unam.mx',
        user='JimmyNeutron',
        password='CarlSheen',
        database='LIMONADA',
        connection_timeout=3
    )
    sql = 'select * from VENTAS'
    df = pd.read_sql(sql, con)
    print('Conexión exitosa a MySQL UNAM.')
except Exception as e:
    print(f'Aviso: Servidor MySQL remoto no disponible ({e}).')
    print('Cargando dataset de respaldo...')
    fallback_paths = [
        'data/VENTAS.csv',
        'VENTAS.csv',
        'https://raw.githubusercontent.com/fabianalexisbernabecruz-code/portafolio-actuaria-unam/main/data/VENTAS.csv'
    ]
    df = None
    for path in fallback_paths:
        try:
            df = pd.read_csv(path)
            print(f'Datos cargados exitosamente desde: {path}')
            break
        except Exception:
            pass
    if df is None:
        raise RuntimeError('No fue posible cargar el dataset de VENTAS.')

print(df.head())


<font face="Times New Roman">

Como se puede ver la tabla incluye como si era fn de semana o no el día en que se vendió la limonada, si era soleado, caluroso, era quincena y el precio. Note que es un conjunto de datos bastante pequeño y escueto.

</font>

<font face="Times New Roman">

# **Entrenar a la red neronal**

</font>

<font face="Times New Roman">

Al desarrollar proyectos es útil tener una GPU potente y para ello tenemos lo siguiente

</font>

In [ ]:
# Verificamos si hay una
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('A {} device was detected.'.format(device))

# Lo imprimimos si lo encontramos
if device=='cuda':
    print (torch.cuda.get_device_name(device=device))

<font face="Times New Roman">

Antes de entrenar a la red necesitamos hacer un poco más fáciles las cosas para la red neuronal

</font>

In [ ]:
# Calculamos la media y la desviación estándar del precio con el fin de que estandaricemos
# Estandarizaremos aquellas columnas que no se encuentran en un valor en (-1,1) pues la red funcionará mejor
# Usamos la variable df de el apartado anterior y las funciones que nos aportan las bibliotecas
priceMean = df["precio"].mean()
priceStd = df["precio"].std()
df['precio'] = (df['precio']-priceMean)/priceStd

numSoldMean = df["vendidos"].mean()
numSoldStd = df["vendidos"].std()
df['vendidos'] = (df['vendidos']-numSoldMean)/numSoldStd

<font face="Times New Roman">

Mostramos la tabla *nueva* estandarizada con valores entre -1 y 1

</font>

In [ ]:
df.head()

<font face="Times New Roman">

Ahora por la teoría que conocemos de redes neuronales sabemos que una red debe tener unos inputs de entradada para las neuronas que através de unos cálculos nos arrojará un valor, en este caso a estos inputs serán todos los apartados de la tabla `VENTAS` a excepción, claro, de `id_venta` y de el valor que queremos predecir a este lo consideramos el `output` que es la columna `vendidos`.

</font>

In [ ]:
# Creamos nuestros tensores de PyTorch
# Extraemos las entradas y creamos un tensor x de PyTorch que serán nuestras entradas (inputs)
inputs = ["fin_semana",	"soleado",	"caluroso",	"quincena",	"precio"]
x = torch.tensor(df[inputs].values, dtype=torch.float, device=device)

# Hacemos lo análogo para las salidas (outputs)
outputs = ["vendidos"]
y = torch.tensor(df[outputs].values,dtype=torch.float, device=device)

<font face="Times New Roman">

Imprimimos los primeros 5 entradas y 5 primeras salidas, claramente cada registro x tiene 5 entradas y el vector y tiene una sola "coordenada"

</font>

In [ ]:
print(x[0:5])
print("\n")
print(y[0:5])

<font face="Times New Roman">

Ahora creamos como tal a la red neuronal que nos ayudará con nuestro propósito de predicción

</font>

In [ ]:
# Importamos nn de neuronal networks para crear el modelo de la red
import torch.nn as nn

# # Definición del modelo Sequential
model = nn.Sequential(
    nn.Linear(5, 100),   # Capa lineal con 5 entradas y 100 salidas
    nn.ReLU(),           # Función de activación ReLU (suele ser la más eficiente)
    nn.Linear(100, 1))   # Capa lineal con 100 entradas y 1 salida

# Lo movemos al servicio disponible
model.to(device)

<font face="Times New Roman">

Ahora pasamos al título de la sección: al entrenamiento de la red neuronal

</font>

In [ ]:
# Importamos optim que nos permite implementaciones de varios algoritmos de optimización
# utilizados para ajustar los parámetros de los modelos durante el entrenamiento.
import torch.optim as optim
# Medimos nuestra red neuronal mediante el error cuadrático medio
criterion = torch.nn.MSELoss()

# Entrenamos nuestra red con un enfoque SGD (Stochastic Gradient Descent) simple
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)  # lr es la tasa de aprendizaje

# Entrenamos nuestra red utilizando todo el conjunto de datos 5 veces
for epoch in range(50):
    totalLoss = 0
    for i in range(len(x)):

        # Pase hacia adelante individual
        ypred = model(x[i])

        # Medimos qué tan bien el modelo predijo versus el valor real
        loss = criterion(ypred, y[i])

        # Seguimos el desempeño del modelo
        totalLoss += loss.item()

        # Actualizamos la red neuronal
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    # Imprimimos nuestra pérdida después de cada iteración de entrenamiento
    print("Pérdida Total: ", totalLoss)


<font face="Times New Roman">

# **Predicciones**

Una vez entrenada la red neuronal, lo que podemos hacer ahora es dar una predicción de las posibles ventas de limonadas

</font>

In [ ]:
# Grafica de predicciones v.s. valores reales
@torch.no_grad()
def graficarPredicciones(modelo, x, y, valorMinimo, valorMaximo):

    model.eval()                               # Establecemos el modelo en modo de inferencia

    predicciones = []                           # Seguimos las predicciones
    reales = []                                # Seguimos las etiquetas reales

    x.to(device)
    y.to(device)
    model.to(device)

    for i in range(len(x)):

        # Pase hacia adelante individual
        pred = model(x[i])

        # Desnormalizamos nuestra predicción
        pred = pred * numSoldStd + numSoldMean
        act = y[i] * numSoldStd + numSoldMean

        # Guardamos la predicción y la etiqueta actual
        predicciones.append(pred.tolist())
        reales.append(act.item())

    # Grafica reales vs. predicciones
    plt.scatter(reales, predicciones)
    plt.xlabel('Ventas reales de limonadas')
    plt.ylabel('Ventas de limonadas predichas')
    plt.plot([valorMinimo, valorMaximo], [valorMinimo, valorMaximo])
    plt.xlim(valorMinimo, valorMaximo)
    plt.ylim(valorMinimo, valorMaximo)

    # Hacemos que la pantalla sea igual en ambas dimensiones
    plt.gca().set_aspect('equal', adjustable='box')
    plt.show()



In [ ]:
graficarPredicciones(model, x, y, 0, 300)

<font face="Times New Roman">

Se puede aprecir que la red es capaz de predecir los valores que queremos con una proximidad al dato real muy muy buena, en términos matemáticos y geométricos encontró la recta que mejor aproxima al conjunto de datos.

</font>

<font face="Times New Roman">

#  **Creación de registro en documento tipo *JSON***

</font>

<font face="Times New Roman">

Ya con los datos reales y las predicciones (como se muestra en la grafica) generamos un documento llamado *Predicciones.json*.

</font>

In [ ]:
@torch.no_grad()
def generarDatosJSON(modelo, x, y, minValue, maxValue, nombre_archivo='predicciones.json'):

    model.eval()  # Establecemos el modelo en modo de inferencia

    predicciones = []  # donde guardaremos las predicciones

    x.to(device)
    y.to(device)
    model.to(device)
    # Lo que tenemos en el paso anterior, pero guardandolo en un documento
    for i in range(len(x)):

        # Pase hacia adelante individual
        pred = model(x[i])

        # Desnormalizamos nuestra predicción
        pred = pred * numSoldStd + numSoldMean
        act = y[i] * numSoldStd + numSoldMean

        # Guardamos la predicción siguiendo el formato solicitado
        predicciones.append({
            "dato": i,
            "red neuronal": [pred.item()],  # Convertimos el tensor a un valor numérico
            "dato real": act.item()
        })

    # Almacenamos las predicciones en un archivo JSON
    with open(nombre_archivo, 'w') as json_file:
        json.dump({"Predicciones": predicciones}, json_file, indent=2)

    print(f"Datos almacenados en {nombre_archivo}")

# Uso de la función para generar y almacenar datos en formato JSON
generarDatosJSON(model, x, y, 0, 300, nombre_archivo='Predicciones.json')

<font face="Times New Roman">

# **Sección Extra**

</font>

<font face="Times New Roman">

Dado que esto anterior es solo el comienzo incluimos:
Utilizar datos reales de algún fenomeno para la predicción de el volumen de ventas de un documento csv de TESLA.

</font>

In [ ]:
url = 'https://raw.githubusercontent.com/FabianABC0/BaseProyectoFinalMDD/main/TSLA.csv'
df = pd.read_csv(url)
print()

In [ ]:
# Verificamos si hay una GPU disponible
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Se detectó un dispositivo {}.'.format(device))

# Imprimimos el nombre del dispositivo si es una GPU
if device == 'cuda':
    print(torch.cuda.get_device_name(device=device))

# Cargar el conjunto de datos desde el repositorio
url = 'https://raw.githubusercontent.com/FabianABC0/BaseProyectoFinalMDD/main/TSLA.csv'
df = pd.read_csv(url)

if df is not None:
    # Normalizamos las columnas 'Open', 'High', 'Low', 'Close', y 'Volume'
    for column in ['Open', 'High', 'Low', 'Close', 'Volume']:
        mean_value = df[column].mean()
        std_value = df[column].std()
        df[column] = (df[column] - mean_value) / std_value

    # Creamos tensores de PyTorch
    inputs = ["Open", "High", "Low", "Close"]
    x = torch.tensor(df[inputs].values, dtype=torch.float, device=device)

    outputs = ["Volume"]
    y = torch.tensor(df[outputs].values, dtype=torch.float, device=device)

    model = nn.Sequential(
        nn.Linear(4, 1000),
        nn.ReLU(),
        nn.Linear(1000, 1)
    )
    # Movemos el modelo al dispositivo disponible


    # # Aumentamos la complejidad del modelo agregando más capas ocultas
    # model = nn.Sequential(
    #     nn.Linear(4, 1000),
    #     nn.ReLU(),
    #     nn.Linear(1000, 500),
    #     nn.ReLU(),
    #     nn.Linear(500, 100),
    #     nn.ReLU(),
    #     nn.Linear(100, 1)
    # )
    model.to(device)

    # Definimos la función de pérdida y el optimizador
    criterion = torch.nn.MSELoss()
    optimizer = optim.SGD(model.parameters(), lr=0.0009, momentum=0.9)  # Ajustamos la tasa de aprendizaje

    # Entrenamos el modelo
    for epoch in range(50):
        total_loss = 0
        for i in range(len(x)):
            y_pred = model(x[i])
            loss = criterion(y_pred, y[i])
            total_loss += loss.item()
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        print("Pérdida Total: ", total_loss)

    # Visualizamos predicciones vs. valores reales
    @torch.no_grad()
    def graphPredictions(model, x, y, min_value, max_value):
        model.eval()
        predictions = []
        actual = []
        x.to(device)
        y.to(device)
        model.to(device)
        for i in range(len(x)):
            pred = model(x[i])
            pred = pred.item()  # Convertimos el tensor a un valor numérico
            act = y[i].item()  # Convertimos el tensor a un valor numérico
            predictions.append(pred)
            actual.append(act)

        plt.scatter(actual, predictions)
        plt.xlabel('Volume real')
        plt.ylabel('Volume predicho')
        plt.plot([min_value, max_value], [min_value, max_value])
        plt.xlim(min_value, max_value)
        plt.ylim(min_value, max_value)
        plt.gca().set_aspect('equal', adjustable='box')
        plt.show()

    graphPredictions(model, x, y, df['Volume'].min(), df['Volume'].max())

    # Generamos y almacenamos datos en formato JSON
    @torch.no_grad()
    def generateDataJSON(model, x, y, filename='predictions.json'):
        model.eval()
        predictions = []
        x.to(device)
        y.to(device)
        model.to(device)
        for i in range(len(x)):
            pred = model(x[i])
            pred = pred.item()  # Convertimos el tensor a un valor numérico
            act = y[i].item()  # Convertimos el tensor a un valor numérico
            predictions.append({
                "dato": i,
                "red neuronal": [pred],
                "dato real": act
            })

        with open(filename, 'w') as json_file:
            json.dump({"Predictions": predictions}, json_file, indent=2)

        print(f"Datos almacenados en {filename}")

    # Usamos la función para generar y almacenar datos en formato JSON
    generateDataJSON(model, x, y, filename='Predictions.json')

else:
    print("No se pudo cargar el conjunto de datos.")


<font face="Times New Roman">

La red dada esa topología logra predecir solo unos pocos por lo que no encontró una solución con esa estructura e hiperparámetros.

</font>

<font face="Times New Roman">

# **Conclusiones**

</font>

<font face="Times New Roman">

En conclusión las redes neuronales son una herramienta muy amplia y cuyo fundamento recae en las ideas matemáticas de la optimización de funciones y del Cálculo en varias variables que puede ser aplicada desde algo tan "básico" como esta red neuronal usada para la predicción de el valor de una variable que nos interesa, como las ventas de una empresa, hasta cosas como la conducción autónoma. En este proyecto pudimos lograr el objetivo de que la red neuronal tomara como parámetros de entrada las primeras columnas de nuestra tabla VENTAS de la base de datos y mediante el aprendizaje lograra acercarce más y más a lograr predecir las ventas (que fue lo último que mostramos antes de la sección extra), por otra parte no se logró hacer lo análogo para la base de datos de las ventas TESLA stock data pese a variar los hiperparámetros de el learning rate e incluso probando otras funciones de activación u otra topología de la red sino que solo la red logró distar de los valores reales en algunos miles que no es deseable, lo que se quería era lo mismo que en la primera parte, una minimización muy buena de la funcion de costos lo que implica una buena estimación.
Para terminar este proyecto basta recalcar que el proyecto estuvo fuertemente basado del tutorial al que se hacer referencia al inicio del Notebook.

</font>

<font face="Times New Roman">

# **Referencias**

</font>

<font face="Times New Roman">

Predict sales using a neural network - LeakyIA. https://www.leaky.ai/projectlemonade

</font>

<font face="Times New Roman">

APRENDE Qué son las Redes Neuronales? Dot CSV. https://youtube.com/playlist?list=PL-Ogd76BhmcB9OjPucsnc2-piEE96jJDQ&si=EyzUnmXIF-ByHITp

</font>

<font face="Times New Roman">

TESLA Stock Data. (2022, 25 marzo). Kaggle. https://www.kaggle.com/datasets/varpit94/tesla-stock-data-updated-till-28jun2021/

Los datos para la sección extra fueron extraidos de este sitio.

</font>